In [ ]:
import os
import shutil
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF

from medicalDataLoader_Enes import MedicalImageDataset, list_png
from utils_Enes import device, evaluate, keep_largest_component
from UNet_Base_Enes import UNet

In [ ]:
MODEL_PATH = 'models/mean_teacher_final/last.pth'
TEST_DIR = ''
OUT_DIR = 'predictions'

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
net = UNet(num_classes=4).to(device)
net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
net.eval()

val_loader = DataLoader(MedicalImageDataset('val', DATA_DIR), batch_size=8)
dice, hd = evaluate(net, val_loader, post_process=True)
print('Dice VD {:.3f} Myo {:.3f} VG {:.3f} (moy. {:.3f}) | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(*dice, np.mean(dice), *hd))

In [ ]:
@torch.no_grad()
def save_predictions(img_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for path in list_png(img_dir):
        image = TF.to_tensor(Image.open(path).convert('L')).unsqueeze(0)
        pred = keep_largest_component(net(image.to(device)).argmax(dim=1).cpu())[0]
        Image.fromarray((pred.numpy() * 85).astype(np.uint8)).save(os.path.join(out_dir, os.path.basename(path)))


save_predictions(os.path.join(DATA_DIR, 'val', 'Img'), os.path.join(OUT_DIR, 'val'))
print(len(os.listdir(os.path.join(OUT_DIR, 'val'))), 'masques enregistrés')

if TEST_DIR:
    save_predictions(TEST_DIR, os.path.join(OUT_DIR, 'test'))
    shutil.make_archive('predictions_test', 'zip', os.path.join(OUT_DIR, 'test'))